## Generate dataset for multiplication only

- a from [0, 1000]
- b from [0, 1000]

Format is JSON:
```
[
    {
       input: [a, b],
       target: a * b
    }
    ...
    ...
]
```

In [1]:
import random
import json

dataset = []

unique_size = 20_000
unique_max_num = 1000

repeat_size = 5_000
repeat_max_num = 100

rand_dist_mu = 4.2
rand_dist_sigma = 1.0

for _ in range(repeat_size):
    a = random.randint(0, repeat_max_num)
    b = random.randint(0, repeat_max_num)
    target = a * b
    data = {
        'input': { 'a': str(a), 'b': str(b) },
        'target': str(target)
    }
    dataset.append(data)

for _ in range(unique_size):
    a = random.randint(0, unique_max_num)
    b = random.randint(0, unique_max_num)
    target = a * b
    data = {
        'input': { 'a': str(a), 'b': str(b) },
        'target': str(target)
    }
    dataset.append(data)

print(json.dumps(dataset[:5], indent=4))
print(f"completed dataset creation of size {len(dataset)}....")

[
    {
        "input": {
            "a": "31",
            "b": "11"
        },
        "target": "341"
    },
    {
        "input": {
            "a": "86",
            "b": "87"
        },
        "target": "7482"
    },
    {
        "input": {
            "a": "3",
            "b": "78"
        },
        "target": "234"
    },
    {
        "input": {
            "a": "74",
            "b": "50"
        },
        "target": "3700"
    },
    {
        "input": {
            "a": "98",
            "b": "94"
        },
        "target": "9212"
    }
]
completed dataset creation of size 25000....


In [10]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from tokenizer import DigitTokenizer
import matplotlib.pyplot as plt
from IPython.display import clear_output, display

CHECKPOINT_DIR = "/teamspace/studios/this_studio/SAIR-competition-modular-math/checkpoints"
DATA_DIR = "/teamspace/studios/this_studio/SAIR-competition-modular-math/data"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")

rand_seed = 42

batch_size = 64
max_seq_len = 12 # very small context window...
d_model = 512
num_heads = 4
lr = 1e-4

Training on device: cpu


In [25]:
tokenizer = DigitTokenizer()
eos_id = tokenizer.char_to_int["<eos>"]
pad_id = tokenizer.char_to_int["<pad>"]
tokenized_input = []

for sample in dataset:
    input_obj = sample["input"]
    str_tgt = sample["target"]
    str_a = input_obj['a']
    str_b = input_obj['b']

    a_token_id = tokenizer.encode(str_a)
    b_token_id = tokenizer.encode(str_b)
    tgt_token_id = tokenizer.encode(str_tgt)
    # KEY DECISION: reversing each input token so QKt aligns during math operations digit wise, from small to big
    a_token_id.reverse(), b_token_id.reverse(), tgt_token_id.reverse()
    tgt_token_id.append(eos_id)

    if len(a_token_id) < max_seq_len:
        a_token_id = a_token_id + [pad_id] * (max_seq_len - len(a_token_id))
    else:
        a_token_id = a_token_id[:max_seq_len]
    
    if len(b_token_id) < max_seq_len:
        b_token_id = b_token_id + [pad_id] * (max_seq_len - len(b_token_id))
    else:
        b_token_id = b_token_id[:max_seq_len]
    
    if len(tgt_token_id) < max_seq_len:
        tgt_token_id = tgt_token_id + [pad_id] * (max_seq_len - len(tgt_token_id))
    else:
        tgt_token_id = tgt_token_id[:max_seq_len]

    tokenized_input.append([a_token_id, b_token_id, tgt_token_id])


dataset_tensor = torch.tensor(tokenized_input, dtype=torch.long)

print(f"Dataset securely loaded in-memory. Matrix shape: {dataset_tensor.shape}")
print(f"The <eos> token id is: {tokenizer.char_to_int['<eos>']}")
print(dataset_tensor[:3])

Dataset securely loaded in-memory. Matrix shape: torch.Size([25000, 3, 12])
The <eos> token id is: 14
tensor([[[ 1,  3, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13],
         [ 1,  1, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13],
         [ 1,  4,  3, 14, 13, 13, 13, 13, 13, 13, 13, 13]],

        [[ 6,  8, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13],
         [ 7,  8, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13],
         [ 2,  8,  4,  7, 14, 13, 13, 13, 13, 13, 13, 13]],

        [[ 3, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13],
         [ 8,  7, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13],
         [ 4,  3,  2, 14, 13, 13, 13, 13, 13, 13, 13, 13]]])


In [26]:
tokenized_standard_input = []

for sample in dataset:
    input_obj = sample["input"]
    str_tgt = sample["target"]
    str_a = input_obj['a']
    str_b = input_obj['b']
    
    standard_str_input = f"{str_a}*{str_b}={str_tgt}"
    standard_encoded_input = tokenizer.encode(standard_str_input)
    standard_encoded_input.append(eos_id)
    
    if len(standard_encoded_input) < 64:
        standard_encoded_input = standard_encoded_input + [pad_id] * (64 - len(standard_encoded_input))
    else:
        standard_encoded_input = standard_encoded_input[:64]

    tokenized_standard_input.append(standard_encoded_input)


standard_dataset_tensor = torch.tensor(tokenized_standard_input, dtype=torch.long)

print(f"Standard dataset securely loaded in-memory. Matrix shape: {standard_dataset_tensor.shape}")
print(f"The <eos> token id is: {tokenizer.char_to_int['<eos>']}")
print(standard_dataset_tensor[:3])

Standard dataset securely loaded in-memory. Matrix shape: torch.Size([25000, 64])
The <eos> token id is: 14
tensor([[ 3,  1, 10,  1,  1, 12,  3,  4,  1, 14, 13, 13, 13, 13, 13, 13, 13, 13,
         13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13,
         13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13,
         13, 13, 13, 13, 13, 13, 13, 13, 13, 13],
        [ 8,  6, 10,  8,  7, 12,  7,  4,  8,  2, 14, 13, 13, 13, 13, 13, 13, 13,
         13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13,
         13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13,
         13, 13, 13, 13, 13, 13, 13, 13, 13, 13],
        [ 3, 10,  7,  8, 12,  2,  3,  4, 14, 13, 13, 13, 13, 13, 13, 13, 13, 13,
         13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13,
         13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13,
         13, 13, 13, 13, 13, 13, 13, 13, 13, 13]])


In [6]:
'''
Using standard GPT2 transformer model as baseline
'''
from transformers import GPT2Config, GPT2LMHeadModel

config = GPT2Config(
    vocab_size=tokenizer.vocab_size,   
    n_embd=512,                       
    n_layer=2,                         
    n_head=4,                          
    n_ctx=64,                        
    bos_token_id=None,                 
    eos_token_id=tokenizer.char_to_int["<eos>"], 
    pad_token_id=tokenizer.char_to_int["<pad>"]
)

baseline_model = GPT2LMHeadModel(config)

num_params = sum(p.numel() for p in baseline_model.parameters() if p.requires_grad)
print(f"Total Trainable Parameters: {num_params:,}")

Total Trainable Parameters: 6,838,272


In [38]:
'''
Just trying to see if this setup can even learn simple multiplication first...
'''
from rope import RoPE

d_k = d_model // num_heads
rope = RoPE(
    d_head=d_k,
    max_seq_len=max_seq_len
)
embedding = nn.Embedding(num_embeddings=tokenizer.vocab_size, embedding_dim=d_model)
pre_attention_layernorm = nn.LayerNorm(d_model)
q_linear = nn.Linear(d_model, d_model)
k_linear = nn.Linear(d_model, d_model)
ffn = nn.Sequential(
            nn.Linear(max_seq_len, 2 * max_seq_len),
            nn.ReLU(),                                 
            nn.Linear(2 * max_seq_len, max_seq_len)
        )
final_layernorm = nn.LayerNorm(max_seq_len)
output_linear = nn.Linear(max_seq_len, tokenizer.vocab_size)



# single run with 1 sample for training
sample = dataset_tensor[0]
a, b, tgt = sample

a_embed = embedding(a)
b_embed = embedding(b)

a_norm1 = pre_attention_layernorm(a_embed)
b_norm1 = pre_attention_layernorm(b_embed)

q = q_linear(a_norm1).view(max_seq_len, num_heads, d_k).transpose(0, 1)
k = k_linear(b_norm1).view(max_seq_len, num_heads, d_k).transpose(0, 1)

q, k = rope(q, k)
score = torch.matmul(q, k.transpose(-2, -1)) / (d_k ** 0.5)
# explicitly skip softmax, and v ....
ffn_out = ffn(score)
final_norm = final_layernorm(ffn_out)
logits = output_linear(final_norm)
